# Unsupervised Learning - Customer Segmentation & Behavioral Profiling

## 1. Introduction

Within the broader scope of the Credit Card Transaction analysis, supervised learning frameworks have been established to detect risk (**Fraud Detection Module**) and forecast immediate needs (**Next Purchase Prediction Module**). However, a critical strategic necessity remains: the derivation of latent customer archetypes.

While supervised models excel at predicting *specific events* (e.g., fraud occurrences or transaction categories), they typically process customers as isolated feature vectors without behavioral context. This study utilizes **Unsupervised Learning** to uncover intrinsic structures within the data, grouping customers based on lifestyle patterns and behavioral dynamics rather than solely on monetary magnitude.

**Strategic Implications:**
Critically, these behavioral insights serve as a blueprint for **precision marketing**. By translating abstract clusters into targeted interventions, discretionary spending can be systematically encouraged and customer loyalty reinforced, serving as a proactive mechanism for **churn mitigation**.

### 1.2. Objectives & Methodology

To derive actionable and scientifically robust customer segments, a four-stage methodological framework was adopted:

#### **Phase 1: Advanced Feature Engineering**
The analysis transcends static wealth metrics by engineering dynamic behavioral features designed to capture lifestyle patterns. Key indicators include:
* **Geospatial Radius of Gyration:** To test the hypothesis of mobility-based segmentation (e.g., distinguishing "Locals" from "Travelers").
* **Spending Entropy:** To quantify the diversity and predictability of purchasing habits.
* **Temporal Ratios:** To measure consumption preferences across time domains (e.g., Weekend vs. Weekday, Night vs. Day).

#### **Phase 2: Multi-Model Benchmarking**
Three distinct algorithmic families were evaluated to determine the optimal structural fit for high-dimensional behavioral data:
1.  **Method 1: K-Means (Partitioning):** Evaluated for the creation of distinct, non-overlapping personas suitable for precise marketing targeting.
2.  **Method 2: Gaussian Mixture Models (Probabilistic):** Tested to assess the hypothesis of "soft" segmentation and determine if customers exhibit overlapping behavioral probabilities.
3.  **Method 3: HDBSCAN (Density-Based):** Utilized to identify dense behavioral cores and detect "Noise" (outliers) that do not conform to standard profiles.

#### **Phase 3: Model Selection Strategy**
All models were rigorously evaluated based on **Silhouette Scores**, **Interpretability**, and **Stability**.
* *Outcome:* The best-performing algorithm (**K-Means**) was selected for detailed profiling and business application.
* *Validation:* The density-based algorithm (**HDBSCAN**) was retained as a secondary validation layer to confirm the structural integrity of the clusters and identify anomalies.

#### **Phase 4: Dynamic Analysis**
The investigation extends beyond static clustering to assess **Temporal Stability**. By comparing customer segments across distinct time periods (e.g., Q1 vs. Q4), the analysis determines whether high-value behaviors represent consistent habits or transient anomalies.

### 1.3. Integration with Project Modules
* **Input Data Hygiene:** To ensure behavioral profiles reflect genuine lifestyle patterns, the dataset is filtered to exclude fraudulent transactions. Fraudulent patterns (e.g., rapid cross-border spending) would statistically mimic "Traveler" behavior, introducing significant noise.
* **Output Utility:** The personas identified in this module serve as macro-level label encodings (feature injection) for downstream supervised models (e.g., a Next-Purchase Prediction module), enhancing the predictive feature space with user-level behavioral context.

## 2. Import Libraries, data and build helper functions

To facilitate the behavioral segmentation analysis, we initialize the computational environment by importing libraries categorized by their specific analytical functions.

**Library Overview:**
* **Data Manipulation:** `pandas` and `numpy` are utilized for vectorization and dataframe management.
* **Visualization:** `matplotlib` and `seaborn` are employed for high-dimensional plotting, while `math.pi` supports the construction of Radar Charts for persona profiling.
* **Statistical Engineering:** `scipy.stats.entropy` is imported specifically to calculate **Spending Entropy**, a key complexity feature.
* **Machine Learning:**
    * **Clustering Algorithms:** `KMeans` (Partitioning), `GaussianMixture` (Probabilistic), and `HDBSCAN` (Density-based) are imported for multi-model benchmarking.
    * **Preprocessing & Validation:** `StandardScaler` for feature normalization, `PCA` for dimensionality reduction/visualization, and `silhouette_score` for model evaluation.
    * **Interpretability:** `DecisionTreeClassifier` and `plot_tree` are used to build surrogate models that translate complex clusters into readable decision rules.

**Data Source:**
The analysis utilizes the **Credit Card Transactions Dataset** sourced via the Kaggle API. The pipeline handles the automated download, extraction, and ingestion of the zipped CSV file.

In [ ]:
!kaggle datasets download -d priyamchoksi/credit-card-transactions-dataset

import os
import warnings
import folium
import zipfile
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from math import pi
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree

zip_file_name = 'credit_card_transactions.csv.zip'
with zipfile.ZipFile(zip_file_name, 'r') as z:
    z.extractall('data') # Extracts to a subfolder named 'data'
    
# Load the specific file from the extracted data
df = pd.read_csv("data/credit_card_transactions.csv")
df.head()

### 2.1. Data Ingestion & Hygiene
The `load_data` function handles the ingestion of raw transaction records. Crucially, it enforces data hygiene by filtering out fraudulent transactions (`is_fraud == 1`). This step is essential to ensure that the subsequent behavioral segmentation reflects genuine consumer lifestyle patterns rather than anomalous theft behaviors, which would otherwise introduce significant noise into the clustering algorithms.

In [ ]:
# 1. Fix KMeans memory leak warning
os.environ["OMP_NUM_THREADS"] = "4"

# 2. Fix the joblib/wmic error
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

# 3. Silence the clutter
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ==========================================
# 1. LOAD DATA
# ==========================================

def load_data(path):
    # Load the CSV
    df = pd.read_csv("data/credit_card_transactions.csv")
    
    # CRITICAL: Filter out fraud transactions (is_fraud == 1)
    # We only want to cluster TRUE customer behavior.
    initial_len = len(df)
    df = df[df['is_fraud'] == 0].copy()
    filtered_len = len(df)
    
    print(f"✅ Data Loaded. Dropped {initial_len - filtered_len} fraud transactions.")
    print(f"   Remaining Clean Transactions: {filtered_len}")
    
    return df

### 2.2. Geospatial Helper Functions
To quantify customer mobility, we implement two vectorized geospatial utilities:
* **`vectorized_haversine`**: Calculates the great-circle distance between two points on a sphere (customer home location vs. merchant location) given their longitudes and latitudes.
* **`compute_rog` (Radius of Gyration)**: A statistical measure of the dispersion of a customer's transaction locations relative to their center of mass. This metric is pivotal for distinguishing "Local" customers (low RoG) from "Travelers" (high RoG).

In [ ]:
# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def vectorized_haversine(lat1, lon1, lat2, lon2):
    R = 6371  #approximate mean radius of the Earth in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def compute_rog(transactions):
    lat_c = transactions['merch_lat'].mean()
    lon_c = transactions['merch_long'].mean()
    d = vectorized_haversine(transactions['merch_lat'], transactions['merch_long'], lat_c, lon_c)
    return np.sqrt((d**2).mean())


## 3. Behavioral Feature Engineering

#### **Philosophy: Decoupling Magnitude from Behavior**
Conventional RFM (Recency, Frequency, Monetary) models often exhibit a bias toward wealth in financial segmentation. For instance, a high-net-worth individual purchasing exclusively essentials may be behaviorally indistinguishable from a student with identical purchasing patterns, yet a magnitude-based model would likely segregate them based solely on transaction volume.

To resolve this, the feature engineering process was designed to decouple **Wealth** (Magnitude) from **Lifestyle** (Behavioral Patterns), ensuring that the resulting personas reflect intrinsic consumption habits rather than financial capacity.

#### **3.1. Geospatial Engineering (Mobility Profiles)**
* **Features:** `dist_km` (Distance from Home), `radius_of_gyration` (RoG).
* **Rationale:** Raw geospatial coordinates (`lat`/`long`) are unsuitable for direct clustering. Instead, these coordinates are transformed into the **Radius of Gyration**, a metric derived from physics that quantifies the spatial dispersion of a customer's transaction history relative to their center of mass.
    * *Low RoG:* Indicative of a "Local" profile (transactions are tightly clustered around the residence).
    * *High RoG:* Indicative of a "Traveler" or "Commuter" profile (transactions are widely dispersed).
* **Technical Implementation:**
    * **Vectorization:** Given the dataset size (1.2 million rows), iterative distance calculations are computationally prohibitive. The Haversine formula was therefore implemented using `numpy` vectorization, reducing computational complexity and enabling efficient processing of large-scale geospatial data.

#### **3.2. Entropy & Complexity (Loyalty Metrics)**
* **Features:** `cat_entropy` (Shannon Entropy of Categories).
* **Rationale:** Simple frequency counts (e.g., "Unique Merchants") are inherently correlated with transaction volume. To measure behavioral predictability independent of volume, **Shannon Entropy** ($H = -\sum p_i \log p_i$) is calculated for merchant categories.
    * *Low Entropy (~0):* Suggests habitual behavior and high loyalty (e.g., repeated patronage of specific merchants).
    * *High Entropy (>2):* Suggests exploratory behavior and low loyalty (e.g., transactions across a diverse set of merchants).
* **Technical Implementation:**
    * **Numerical Stability:** The `scipy.stats.entropy` function is utilized to compute entropy over the probability distribution of merchant categories, ensuring robust handling of sparse vectors.

#### **3.3. Temporal & Category Ratios (Lifestyle Fingerprints)**
* **Features:** `weekend_ratio`, `night_ratio`, `grocery_ratio`, `gas_ratio`.
* **Rationale:** Ratios serve as the primary differentiators for behavioral segmentation. By normalizing specific category spend against total spend, wealth bias is eliminated.
    * *Example:* A `gas_ratio` of 0.50 identifies a "Commuter" profile, regardless of whether the absolute spend is \$100 or \$10,000.
    * *Example:* A high `night_ratio` distinguishes "Socializers" or "Shift Workers" from standard "9-to-5" consumers.
* **Technical Implementation:**
    * **Aggregation Strategy:** Pandas' split-apply-combine paradigm is employed to aggregate transaction-level data into normalized customer-level feature vectors efficiently.

#### **3.4. Distribution Transformation (Log-Scaling)**
* **Features:** `log_amt_sum`, `log_dist_km_max`.
* **Rationale:** Financial transaction data typically exhibits a **Power Law (Pareto) distribution**, where extreme outliers (e.g., top 1% of spenders) distort the feature space. Standard distance-based clustering algorithms (such as K-Means) are highly sensitive to such skewness.
* **Technical Implementation:**
    * **Log-Transformation:** A `np.log1p` transformation ($\log(x + 1)$) is applied to all magnitude-based features. This compresses the dynamic range while handling zero values, resulting in a near-normal distribution suitable for `StandardScaler` normalization.

In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================

def create_features(df):
    df = df.copy()

    # 1. PRE-CALCULATIONS
    # --------------------------------------
    # Geospatial
    df['dist_km'] = vectorized_haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
    
    # Time
    df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour'] = df['trans_dt'].dt.hour
    df['day_of_week'] = df['trans_dt'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_night'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)

    # 2. CUSTOMER AGGREGATES
    # --------------------------------------
    
    # A. Magnitude (Wealth & Volume)
    # We use 'amt' for raw spend. 
    magnitude = df.groupby('cc_num').agg({
        'amt': ['sum', 'mean', 'max', 'std'], 
        'dist_km': ['mean', 'max'],
        'trans_dt': 'count'
    })
    # Flatten columns (e.g. 'amt_sum', 'dist_km_max')
    magnitude.columns = ['_'.join(col).strip() for col in magnitude.columns.values]
    magnitude.rename(columns={'trans_dt_count': 'txn_count'}, inplace=True)

    # B. Behavioral Ratios (The "Lifestyle" Fingerprint)
    # Percentage of spend on Weekends
    weekend_spend = df[df['is_weekend']==1].groupby('cc_num')['amt'].sum()
    weekend_ratio = (weekend_spend / magnitude['amt_sum']).rename('weekend_ratio').fillna(0)

    # Percentage of spend at Night
    night_spend = df[df['is_night']==1].groupby('cc_num')['amt'].sum()
    night_ratio = (night_spend / magnitude['amt_sum']).rename('night_ratio').fillna(0)

    # Percentage of spend on Grocery & Gas (Key Differentiators)
    cat_spend = df.groupby(['cc_num', 'category'])['amt'].sum().unstack(fill_value=0)
    # Calculate ratios for key categories only (reduces noise)
    total_spend = magnitude['amt_sum']
    grocery_ratio = (cat_spend.get('grocery_pos', 0) / total_spend).rename('grocery_ratio')
    gas_ratio = (cat_spend.get('gas_transport', 0) / total_spend).rename('gas_ratio')
    dining_ratio = (cat_spend.get('food_dining', 0) / total_spend).rename('dining_ratio')
    travel_ratio = (cat_spend.get('travel', 0) / total_spend).rename('travel_ratio')

    # C. Complexity & Dispersion (Entropy & RoG)
    # Category Entropy (Variety of spend)
    cat_entropy = df.groupby('cc_num')['category'].apply(
        lambda g: entropy(g.value_counts(normalize=True).values + 1e-9)
    ).rename('cat_entropy')

    # Radius of Gyration (How far they roam)
    rog = df[['cc_num','merch_lat','merch_long']].groupby('cc_num').apply(
        lambda g: compute_rog(g), include_groups=False
    ).rename('radius_of_gyration')

    # 3. MERGE & LOG TRANSFORM
    # --------------------------------------
    features = pd.concat([
        magnitude, 
        weekend_ratio, night_ratio, 
        grocery_ratio, gas_ratio, dining_ratio, travel_ratio,
        cat_entropy, rog
    ], axis=1).fillna(0)

    # CRITICAL: Log Transform skewed magnitude columns
    # We leave Ratios (0-1) and Entropy alone. We only log money and distance.
    cols_to_log = ['amt_sum', 'amt_mean', 'amt_max', 'amt_std', 
                   'dist_km_mean', 'dist_km_max', 'txn_count', 'radius_of_gyration']
    
    for col in cols_to_log:
        # log1p handles zeros safely (log(0+1) = 0)
        features[col] = np.log1p(features[col])

    return features

###  Feature Normalization
Since clustering algorithms (particularly K-Means) are sensitive to the scale of input variables, the `scale_features` function applies `StandardScaler` to standardize all features to a mean of 0 and unit variance. This ensures that magnitude-heavy features (like Total Spend) do not disproportionately dominate the distance calculations against ratio-based features.

In [ ]:
# ==========================================
# 4. SCALING
# ==========================================

def scale_features(features):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)
    return scaled, scaler

### Model Architectures
We define wrapper functions for individual clustering algorithms to facilitate modular testing:
* **`run_kmeans`**: Implements the K-Means algorithm (Partitioning), optimizing for variance minimization.
* **`run_gmm`**: Implements Gaussian Mixture Models (Probabilistic), allowing for soft clustering with elliptical boundaries.

### Execution Pipeline
The `pipeline` function serves as the central orchestrator, chaining the data loading, feature creation, and scaling steps into a single executable workflow. This ensures reproducibility and creates the final `X_scaled` matrix required for model benchmarking.

In [ ]:
# ==========================================
# 6. MAIN PIPELINE FUNCTION
# ==========================================

def pipeline(path):
    df = load_data(path)
    features = create_features(df)
    X_scaled, scaler = scale_features(features)

    return df, features, X_scaled, scaler

### Multi-Model Benchmarking Framework
To determine the optimal segmentation strategy, we implement a comparative framework that tests three algorithmic families across a range of clusters ($k=2$ to $6$):
* **Partitioning (K-Means):** Tested for distinct, actionable marketing segments.
* **Probabilistic (GMM):** Evaluated for overlapping behavioral patterns.
* **Density-Based (HDBSCAN):** Utilized to detect noise and validate cluster density.
    * *Note:* HDBSCAN is configured with `min_cluster_size=20` to identify dense micro-clusters and separate them from sparse noise points.

In [ ]:

# =============================================================
# 7. CLUSTERING PIPELINE (KMeans + GMM + HDBSCAN)
# =============================================================


def run_kmeans(X, ks=(2,3,4,5,6)):
    results = {}
    for k in ks:
        km = KMeans(n_clusters=k, random_state=42)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        results[k] = {
            'model': km,
            'labels': labels,
            'silhouette': sil
        }
    return results


def run_gmm(X, ks=(2,3,4,5,6)):
    results = {}
    for k in ks:
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
        labels = gmm.fit_predict(X)
        sil = silhouette_score(X, labels)
        results[k] = {
            'model': gmm,
            'labels': labels,
            'silhouette': sil
        }
    return results


def run_hdbscan(X, min_cluster_size=20, min_samples=None):
    hdb = HDBSCAN(min_cluster_size=min_cluster_size, 
                  min_samples=min_samples, 
                  metric='euclidean')
    
    labels = hdb.fit_predict(X)
    
    valid_idx = labels != -1
    if valid_idx.sum() > 1:
        sil = silhouette_score(X[valid_idx], labels[valid_idx]) 
    else:
        sil = -1
        
    return {
        'model': hdb,
        'labels': labels,
        'silhouette': sil,
        'n_clusters': len(set(labels)) - (1 if -1 in labels else 0)
    }



def compare_clustering_results(X, ks=(2,3,4,5,6)):
    kmeans_results = run_kmeans(X, ks)
    gmm_results = run_gmm(X, ks)
    hdbscan_result = run_hdbscan(X)

    summary = []
    for k in ks:
        summary.append([
            k,
            kmeans_results[k]['silhouette'],
            gmm_results[k]['silhouette']
        ])

    # Add HDBSCAN summary separately
    summary.append(['HDBSCAN', hdbscan_result['silhouette'], None])

    return summary, kmeans_results, gmm_results, hdbscan_result


def plot_clusters(X, labels, title):
    plt.figure(figsize=(8, 6))
    
    # Create a custom color map
    unique_labels = np.unique(labels)
    
    # 1. Plot Noise (Label = -1) in RED
    if -1 in unique_labels:
        noise_mask = (labels == -1)
        plt.scatter(X[noise_mask, 0], X[noise_mask, 1], 
                    c='red', s=5, alpha=0.5, label='Noise (-1)')
    
    # 2. Plot Clusters (Label >= 0) in various colors
    # We use a colormap 'viridis' or 'tab10' for distinct clusters
    cluster_mask = (labels != -1)
    if cluster_mask.sum() > 0:
        plt.scatter(X[cluster_mask, 0], X[cluster_mask, 1], 
                    c=labels[cluster_mask], cmap='viridis', s=8, alpha=0.7)
    
    plt.title(title)
    plt.legend(loc='upper right', markerscale=2)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

### Evaluation & Visualization
* **`compare_clustering_results`**: Aggregates the **Silhouette Scores** for all models into a summary table, enabling a quantitative comparison of cluster separation and cohesion.
* **`plot_clusters`**: Visualizes the resulting segments in a 2D space (typically post-PCA reduction), allowing for a qualitative assessment of cluster boundaries and structure.

In [ ]:
# =============================================================
# 8. MAIN EXECUTION BLOCK
# =============================================================

def run_hdbscan_sklearn(X, min_cluster_size=100, min_samples=5):
    hdb = HDBSCAN(
        min_cluster_size=min_cluster_size, 
        min_samples=min_samples, 
        metric='manhattan', 
        cluster_selection_method='eom' # 'eom' (Excess of Mass) creates bigger, more stable clusters
    )
    
    labels = hdb.fit_predict(X)
    
    # Calculate Score (ignoring noise)
    valid_idx = labels != -1
    if valid_idx.sum() > 1:
        sil = silhouette_score(X[valid_idx], labels[valid_idx]) 
    else:
        sil = -1
        
    return {
        'model': hdb,
        'labels': labels,
        'silhouette': sil,
        'n_clusters': len(set(labels)) - (1 if -1 in labels else 0),
        'noise_ratio': (labels == -1).sum() / len(labels)
    }


## 4. K-Means and GMM

This execution block orchestrates the primary segmentation workflow, integrating feature engineering, dimensionality reduction, and comparative model benchmarking. The pipeline proceeds through the following stages:

**1. Feature Extraction & Standardization**
Raw transaction data is transformed into the behavioral feature set defined in Section 3. To prevent magnitude-based bias (e.g., Total Spend dominating Ratio features), all vectors are normalized using `StandardScaler` ($\mu=0, \sigma=1$) prior to algorithm ingestion.

**2. Dimensionality Reduction (PCA)**
Principal Component Analysis (PCA) is applied to compress the 15-dimensional feature space into 2 principal components. While the clustering algorithms operate on the full high-dimensional dataset (`X_scaled`) to preserve information, the PCA projection (`X_pca`) is generated exclusively to facilitate the 2D visualization of cluster separability.

**3. Comparative Benchmarking (Dual-Metric Validation)**
A rigorous "tournament" style evaluation is conducted between **K-Means** (Partitioning) and **Gaussian Mixture Models** (Probabilistic) across cluster counts of $k=3$ to $k=6$. To ensure robustness, we employ a dual-metric validation approach:
* **The Silhouette Coefficient:** Calculated to quantify the quality of separation and cohesion between clusters.
* **The Elbow Method:** Utilized to plot the Within-Cluster Sum of Squares (Inertia). This visualizes the point of diminishing returns (the "elbow") where adding complexity ($k$) no longer yields a significant drop in variance.



**4. Automated Model Selection**
The pipeline programmatically identifies the optimal segmentation solution. While the primary selection criterion is maximizing the **Silhouette Score**, the **Elbow Plot** serves as a critical visual validation, confirming that the selected $k$ represents a structural inflection point in the data rather than a local anomaly. This ensures the final personas are both mathematically distinct and statistically efficient.

In [ ]:
# =============================================================
# PHASE 1: PRE-PROCESSING & SEGMENTATION (K-Means / GMM)
# =============================================================
if __name__ == "__main__":
    print(f"✅ Data Loaded. Shape: {df.shape}")

    # 1. Run Feature Engineering
    print("🛠️ Creating Features (this may take 1-2 mins)...")
    features_df = create_features(df)
    features_df.dropna(inplace=True)
    
    # 2. Scale Features (Critical for Clustering)
    print("⚖️ Scaling Features...")
    X_scaled, scaler = scale_features(features_df)

    # 3. Run PCA (Required for Visualization)
    print("📉 Running PCA for Visualization...")
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # 4. Benchmarking K-Means vs GMM
    print("🚀 Running Partitioning & Probabilistic Models...")
    test_ks = [3, 4, 5, 6]
    
    kmeans_results = run_kmeans(X_scaled, ks=test_ks)
    gmm_results = run_gmm(X_scaled, ks=test_ks)

    # 5. Print Comparison Table
    print("\n" + "="*40)
    print(f"{'Method':<15} | {'K':<5} | {'Silhouette Score':<15}")
    print("="*40)
    
    for k in test_ks:
        print(f"{'K-Means':<15} | {k:<5} | {kmeans_results[k]['silhouette']:.4f}")
        print(f"{'GMM':<15} | {k:<5} | {gmm_results[k]['silhouette']:.4f}")
    print("="*40 + "\n")

    # 6. AUTO-SELECT BEST K-MEANS
    # Find the K with the max silhouette score
    best_k = max(kmeans_results, key=lambda k: kmeans_results[k]['silhouette'])
    best_score = kmeans_results[best_k]['silhouette']

    print(f"🏆 Best K-Means Model Found: K={best_k} (Score: {best_score:.4f})")
    
    # 7. Visualize Best K-Means
    print(f"🎨 Visualizing Best K-Means Result (K={best_k})...")
    plot_clusters(X_pca, kmeans_results[best_k]['labels'], f"Method 1: Best K-Means (K={best_k})")
    
    # 8. Save K-Means Results
    features_df['Cluster_KMeans'] = kmeans_results[best_k]['labels']
    print(f"✅ K-Means Labels saved to 'features_df'.")

In [ ]:
### 4.1 PCA intepretation
While the PCA scatter plot visualizes the geometric separation of clusters, the axes themselves (PC1 and PC2) are abstract representations. To decode their semantic meaning, this block extracts the **PCA Loadings** (Eigenvectors).

* **Loadings Extraction:** A dataframe is constructed to map the original behavioral features to the two principal components.
* **Driver Identification:** Features are ranked by their absolute coefficient magnitude to identify the primary drivers of variance along each axis. This mathematically proves whether the separation is driven by wealth, mobility, or temporal habits.
* **Visualization:** A heatmap is generated to visualize the correlation strength and direction (positive/negative) between features and components, providing the empirical basis for interpreting the "X" and "Y" axes of the customer map.


In [ ]:
# ==========================================
# OPTIMIZED ELBOW METHOD 
# ==========================================

# 1. Sample the Data (Speed Trick 🚀)
# Calculating inertia on 1.2M rows 10 times is too slow.
# 50,000 rows is statistically enough to see the exact same curve shape.
sample_size = 50000

if len(features_df) > sample_size:
    print(f"📉 Sampling {sample_size} rows for fast Elbow Plot...")
    features_sample = features_df.sample(n=sample_size, random_state=42)
else:
    features_sample = features_df

# 2. Scale the Sample
# (Always scale fresh for the sample to be accurate)
scaler_sample = StandardScaler()
X_sample_scaled = scaler_sample.fit_transform(features_sample)

# 3. Run Elbow Method
wcss = []
K_range = range(1, 11)  # Test K from 1 to 10

print("🔄 Calculating Inertia for K=1 to 10...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_sample_scaled)
    wcss.append(kmeans.inertia_)

# 4. Plot
plt.figure(figsize=(10, 6))
plt.plot(K_range, wcss, marker='o', linestyle='--', color='b', linewidth=2, markersize=8)
plt.title('Elbow Method (Based on 50k Sample)', fontsize=16)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('WCSS (Inertia)', fontsize=12)
plt.xticks(K_range)
plt.grid(True, alpha=0.3)

# Annotate the "Elbow" (usually K=3 or K=4 for this data)
plt.annotate('Likely Elbow (K=3)', xy=(3, wcss[2]), xytext=(4, wcss[2] + 10000),
             arrowprops=dict(facecolor='black', shrink=0.05), fontsize=11)

plt.show()

Elbow method has also confirmed K=3 is a good option.

In [ ]:
# ==========================================
# INTERPRET PCA: WHAT DO THE AXES MEAN?
# ==========================================

# 1. Define Feature Names (Drop labels if they exist)
feature_names = features_df.drop(columns=['Cluster_KMeans', 'Cluster_HDBSCAN'], errors='ignore').columns

# 2. Create Loadings DataFrame
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=['PC1', 'PC2'], 
    index=feature_names
)

# 3. Sort by Absolute Strength, but Print Original Values
# (sort by .abs(), but display the original 'loadings')

print("--- PC1 DRIVERS (The X-Axis) ---")
# Get top 5 indices by absolute magnitude
top_pc1_idx = loadings['PC1'].abs().sort_values(ascending=False).head(5).index
print(loadings.loc[top_pc1_idx, 'PC1'])

print("\n--- PC2 DRIVERS (The Y-Axis) ---")
# Get top 5 indices by absolute magnitude
top_pc2_idx = loadings['PC2'].abs().sort_values(ascending=False).head(5).index
print(loadings.loc[top_pc2_idx, 'PC2'])

# 4. Visualize
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(loadings, cmap='RdBu', center=0, annot=True, fmt=".2f")
plt.title("PCA Loadings: Which Features Created the Map?")
plt.show()


### 4.2. Cluster Interpretation & Profiling

#### **4.2.1 Structural Validation & Model Selection**
To ensure the robustness of the derived segments, a comparative analysis was conducted between **K-Means (Partitioning)** and **Gaussian Mixture Models (Probabilistic)** across a range of cluster counts ($k=3$ to $6$).

**1. Algorithm Benchmarking:**
* **Optimal Convergence (K=3):** Both K-Means and GMM achieved an identical global maximum Silhouette Score of **0.2905**. This rare convergence between a geometric and a probabilistic model provides strong evidence that the three primary customer archetypes are structurally distinct and stable.
* **Performance Divergence:** As complexity increased ($k>3$), GMM performance degraded significantly (Score dropped to **0.15** at K=4), likely due to overfitting the covariance matrices in high-dimensional space. In contrast, K-Means maintained structural integrity (Score **0.23–0.26**).
* **Decision:** Consequently, **K-Means** was selected as the primary segmentation engine for its superior stability and interpretability.

**2. Principal Component Analysis (PCA) Drivers:**
To interpret the geometric separation of the selected K-Means clusters, the PCA Loadings were analyzed:
* **PC1 (The "Velocity" Axis):** Heavily weighted by `txn_count` (+0.39) and `cat_entropy` (+0.39) inversely against `amt_mean` (-0.38). This axis separates users based on **Transaction Velocity**, distinguishing high-frequency/low-value users from low-frequency/high-ticket users.
* **PC2 (The "Mobility" Axis):** Dominated by `dist_km_mean` (+0.58) and `radius_of_gyration` (+0.50). This axis separates **Locals** from **Travelers**, confirming that geospatial behavior is a fundamental differentiator independent of wealth.

**3. Hierarchical Structure (The K=6 Rebound):**
A notable "rebound" in the Silhouette Score was observed at **K=6 (Score: 0.26)** following a decline at K=4 and K=5. This pattern suggests a **hierarchical clustering structure**. While the three primary personas (Macro-Segments) capture the dominant variance, a secondary layer of granularity exists at K=6, likely representing micro-segmentations (e.g., subdividing "Locals" by specific wealth tiers). For strategic clarity, the analysis proceeds with the **K=3 Macro-Segments**.

In [ ]:
# ==========================================
# 1. PREPARE DATA FOR PROFILING
# ==========================================
# We want to look at the MEAN of each feature for each cluster
# Select the most "story-telling" features
profile_cols = [
    'amt_sum',          # Wealth (Log Scale)
    'dist_km_mean',     # Travel (Log Scale)
    'weekend_ratio',    # Worker vs Leisure
    'night_ratio',      # Lifestyle
    'grocery_ratio',    # Family
    'dining_ratio',     # Social
    'gas_ratio'         # Commuter
]

# Group by the Best Cluster (K=3)
cluster_profile = features_df.groupby('Cluster_KMeans')[profile_cols].mean()

# Standardize the data just for the heatmap/radar (Z-score)
# This makes it easy to see "High" (Red) vs "Low" (Blue) relative to average
norm_profile = (cluster_profile - features_df[profile_cols].mean()) / features_df[profile_cols].std()

# ==========================================
# 2. THE HEATMAP (The Quick Check)
# ==========================================
plt.figure(figsize=(12, 5))
sns.heatmap(norm_profile.T, cmap='RdBu_r', center=0, annot=True, fmt=".2f")
plt.title('Cluster DNA: Z-Score Features (Red = High, Blue = Low)')
plt.xlabel('Cluster ID')
plt.show()

# ==========================================
# 3. THE RADAR CHART (The Report Visual)
# ==========================================
def plot_radar(data, title):
    categories = list(data.columns)
    N = len(categories)
    
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    
    # Draw one line per cluster
    palette = ['b', 'r', 'g', 'orange', 'purple', 'cyan']
    
    for i, row in data.iterrows():
        values = row.values.flatten().tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=f'Cluster {i}', color=palette[i % len(palette)])
        ax.fill(angles, values, color=palette[i % len(palette)], alpha=0.1)
        
    plt.xticks(angles[:-1], categories)
    plt.title(title, size=20, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.show()

# Plot using the normalized data so scales match
plot_radar(norm_profile, "Customer Persona Radar")

#### **4.2.2 Behavioral DNA (Heatmap & Radar Analysis)**
To translate the geometric clusters into actionable business personas, the feature centroids (Z-scores) were analyzed via Heatmap and Radar Charts.

* **Feature Analysis:** The Heatmap reveals that the customer base is segmented primarily by **Financial Magnitude**, **Temporal Habits**, and **Essential Engagement**.
    * **Wealth Polarization:** The `amt_sum` feature exhibits a massive divergence. Cluster 1 is characterized by a deep negative Z-score (**-2.73**), representing a distinct "Low-Value" segment, while Clusters 0 and 2 form a "High-Value" tier with positive wealth scores.
    * **Temporal Polarization:** The `night_ratio` feature serves as the singular defining characteristic for Cluster 1, which exhibits an extreme positive spike (**Z = +1.92**). This identifies a behavioral anomaly: a group that is active almost exclusively during non-standard banking hours.
    * **Category Polarization:** Among the high-wealth tier (Clusters 0 and 2), `gas_ratio` acts as the dividing line. Cluster 0 exhibits negligible engagement with vehicle fuel (**Z = -1.18**), whereas Cluster 2 shows positive engagement (**Z = +0.32**), suggesting a lifestyle difference between "Urban/Passive" and "Suburban/Active" mobility.

#### **4.2.3 Final Persona Definitions (Data-Driven)**
Synthesizing the structural (PCA) and behavioral (Heatmap) evidence, three distinct archetypes were defined:

**1. Cluster 0: The Wealthy VIP (The "Hidden" Spender)**
* **Data Signature:** Highest Total Spend (`Z = +0.43`) but significantly low engagement with daily essentials like Gas (`Z = -1.18`) and Groceries (`Z = -0.43`).
* **Behavior:** These high-net-worth individuals utilize the card for high-ticket, non-essential purchases (likely Luxury Retail, Travel, or Services) rather than mundane daily needs. Their spending volume is substantial but concentrated outside the typical "household" categories.
* **Strategic Implication:** **Retention.** This is the highest revenue segment per transaction. Strategy should focus on premium perks (Concierge, Travel Insurance) rather than cashback on essentials.

**2. Cluster 2: The Mainstream Family (The "All-Rounder")**
* **Data Signature:** The only segment exhibiting positive Z-scores across all daily living categories: Gas (`Z = +0.32`), Dining (`Z = +0.17`), and Groceries (`Z = +0.05`).
* **Behavior:** These are active, high-value users who utilize the card as their primary payment method for day-to-day life (Commuting, Eating, Living). They represent the "Core" stable user base.
* **Strategic Implication:** **Share-of-Wallet.** Since engagement is already high frequency, the objective is to consolidate *all* household spending via "Category Bonuses" (e.g., 5% on Gas/Groceries) to prevent leakage to competitor cards.

**3. Cluster 1: The Budget Night Owl (The "Niche")**
* **Data Signature:** Characterized by significantly lower spending power (`Z = -2.73`) but massive nocturnal activity (`Night_Ratio ~ 50%`, `Z = +1.92`).
* **Behavior:** This profile strongly suggests a younger demographic (Students, Gig Workers) or digital-native users (e.g., Gamers) who are active during late hours and spend minimally on traditional dining or fuel.
* **Strategic Implication:** **Activation.** While currently low-value, this segment is highly distinct. Engagement strategies should leverage digital-first offers (Streaming subscriptions, Online Gaming credits) to build loyalty early in the customer lifecycle.

## 5.  Density-Based Structural Validation (HDBSCAN)

Following the primary segmentation via K-Means, the **HDBSCAN** algorithm is deployed as a validation layer to test the structural integrity of the derived clusters. Unlike K-Means, which forces every data point into a centroid-based partition, HDBSCAN identifies clusters based on regions of high data density, designating sparse points as "Noise" (Label -1).

**1. Metric Adaptation (Manhattan Distance)**
To mitigate the "Curse of Dimensionality" inherent in the high-dimensional behavioral feature space, the distance metric is adapted from Euclidean ($L_2$) to **Manhattan ($L_1$)**. While Euclidean distance disproportionately weights large outliers (due to squaring differences), Manhattan distance provides a more robust measure of dissimilarity for sparse behavioral data, resulting in tighter density estimation.

**2. Anomaly Quantification**
A critical output of this phase is the **Noise Ratio**—the percentage of customers that do not belong to any dense cluster. From a business perspective, these unclassified points are not failures of the model but rather represent **behavioral outliers** (e.g., erratic spenders or potential fraud cases missed by supervised filters) that require distinct operational treatment compared to the core segments.

**3. Visualization & Integration**
The resulting density structure is projected onto the PCA space, with noise points explicitly visualized in **Red** to contrast against the dense core clusters. The cluster labels are then integrated into the master dataframe (`features_df`) to facilitate direct comparison with the K-Means personas.

In [ ]:
# =============================================================
# PHASE 2: DENSITY-BASED VALIDATION (HDBSCAN)
# =============================================================

# 1. Run HDBSCAN (Using Manhattan Distance for High Dimensions)
print("🚀 Running Method 3: HDBSCAN (Density Validation)...")

# Note: We use Manhattan distance as discussed for better performance in high-dim
hdbscan_res = run_hdbscan_sklearn(X_scaled, min_cluster_size=100)

# 2. Print Metrics
n_clusters = hdbscan_res['n_clusters']
sil_score = hdbscan_res['silhouette']
noise_ratio = hdbscan_res['noise_ratio']

print("\n" + "="*40)
print(f"HDBSCAN RESULTS (Validation)")
print("="*40)
print(f"Clusters Found : {n_clusters}")
print(f"Silhouette     : {sil_score:.4f}")
print(f"Noise Ratio    : {noise_ratio:.1%} (Points labeled as -1)")
print("="*40 + "\n")

# 3. Visualize HDBSCAN
print("🎨 Visualizing HDBSCAN Result...")
# Red points will indicate Noise (-1)
plot_clusters(X_pca, hdbscan_res['labels'], "Method 3: HDBSCAN (Red = Noise)")

# 4. Save HDBSCAN Results
features_df['Cluster_HDBSCAN'] = hdbscan_res['labels']
print(f"✅ HDBSCAN Labels saved to 'features_df'.")

### 5.1 Structural Validation (HDBSCAN Results)

To test the structural integrity of the K-Means personas, we applied **HDBSCAN (Density-Based Spatial Clustering)** using Manhattan distance.

**Findings:**
1.  **High Noise Ratio (30.2%):**
    HDBSCAN classified ~30% of the customer base as "Noise" (Label -1).
    * *Previous Hypothesis:* We initially hypothesized noise would represent high-mobility travelers.
    * *Data-Driven Correction:* Since K-Means analysis proved that mobility is uniform across clusters, this noise instead represents **Spending Variance**. These are customers whose transaction magnitudes fluctuate too wildly to fit into the stable "Budget" or "VIP" patterns (e.g., a user who spends \$5 one month and \$5,000 the next).

2.  **Core Consolidation (2 Clusters found):**
    HDBSCAN merged the **"Wealthy VIP"** (Cluster 0) and **"Mainstream Family"** (Cluster 2) into a single dense core, while likely keeping the **"Budget Night Owl"** (Cluster 1) or rejecting it as sparse.
    * *Interpretation:* This confirms that the "Mainstream" and "Wealthy" segments exist on a **continuous wealth spectrum**. There is no hard mathematical boundary where "Middle Class" ends and "Rich" begins. K-Means enforces a useful business cutoff, but HDBSCAN reveals the underlying continuity.

**Strategic Decision:**
We retain **K-Means (K=3)** as the primary model because it successfully imposes actionable boundaries on the continuous wealth spectrum, creating distinct tiers for "Retention" vs. "Share-of-Wallet" strategies.


### 6. Advanced Insights: Spatial, Temporal, and Structural Analysis

To transition from abstract mathematical clustering to actionable business intelligence, this section deploys four advanced analytical frameworks:

1.  **Geospatial Visualization:** By projecting cluster labels onto a geographic map, we test the hypothesis of physical mobility. Comparing the widespread dispersion of the "Mainstream" and "VIP" segments against the dense urban cores of the "Night Owl" segment provides physical validation of the behavioral personas.
2.  **Feature Interaction Analysis (Correlation Matrix):** To ensure the stability of the segmentation, we examine the pairwise correlations between features. This reveals the underlying structural dependencies of customer behavior (e.g., the inverse relationship between *Transaction Frequency* and *Ticket Size*) and validates that the personas are driven by distinct, robust signals rather than multicollinear noise.
3.  **Dynamic Stability Analysis (Transition Matrix):** Static clustering offers only a snapshot in time. To assess long-term customer value, we analyze the temporal stability of segments between Q1 (Jan-Mar) and Q4 (Oct-Dec), quantifying the probability of a "High-Value" customer churning to a "Low-Value" segment over time.
4.  **Surrogate Model Explainability:** To bridge the gap between complex high-dimensional clustering and stakeholder interpretability, a **Decision Tree** classifier is trained on the cluster labels. This surrogate model translates the opaque K-Means logic into human-readable "If-Then" rules, providing clear operational criteria for marketing teams.

### 6.1 Geospatial & Density Analysis

In [ ]:
# ==========================================
# GEOSPATIAL VISUALIZATION (WITH LEGEND)
# ==========================================

def plot_cluster_map_with_legend(df, cluster_col, num_samples=1000):
    # 1. Setup Map centered on US
    m = folium.Map(location=[39.8283, -98.5795], zoom_start=4, tiles='CartoDB positron')
    
    # 2. Define Colors & Names for each Cluster
    # We map specific colors to your Personas so they are distinct
    cluster_config = {
        0: {'color': '#E31A1C', 'name': '0: Wealthy VIP (Red)'},       # Red
        1: {'color': '#1F78B4', 'name': '1: Budget Night Owl (Blue)'}, # Blue
        2: {'color': '#33A02C', 'name': '2: Mainstream Family (Green)'} # Green
    }
    
    # 3. Create a "FeatureGroup" for each cluster (This creates the Legend layers)
    layers = {}
    for cid, config in cluster_config.items():
        layers[cid] = folium.FeatureGroup(name=config['name'])

    # 4. Sample data to avoid crashing browser
    sample_df = df.sample(n=min(num_samples, len(df)), random_state=42)
    
    # 5. Plot Points
    for idx, row in sample_df.iterrows():
        cluster = int(row[cluster_col])
        
        # Skip Noise if passing HDBSCAN (-1), or unexpected clusters
        if cluster not in cluster_config: 
            continue
            
        # Add marker to the specific Cluster Layer
        folium.CircleMarker(
            location=[row['merch_lat'], row['merch_long']],
            radius=4, # Slightly larger for visibility
            color=cluster_config[cluster]['color'],
            fill=True,
            fill_color=cluster_config[cluster]['color'],
            fill_opacity=0.6,
            weight=1,
            popup=f"Cluster: {cluster}<br>Amt: ${row['amt']:.2f}"
        ).add_to(layers[cluster])
        
    # 6. Add layers to Map
    for layer in layers.values():
        layer.add_to(m)
        
    # 7. Add the Interactive Legend (Layer Control)
    folium.LayerControl(collapsed=False).add_to(m)
    
    return m

# --- EXECUTION ---
print("Generating Map with Legend... (Check top-right corner of the map!)")

# Ensure Cluster Labels are in the main df (Run the merge code if you haven't already)
if 'Cluster_KMeans' not in df.columns:
    df['Cluster_KMeans'] = df['cc_num'].map(features_df['Cluster_KMeans'])
    df = df.dropna(subset=['Cluster_KMeans'])

map_viz = plot_cluster_map_with_legend(df, 'Cluster_KMeans')
map_viz.save("cluster_map_labeled.html")
map_viz

In [ ]:
# ==========================================
# PLOT HDBSCAN MAP (WITH NOISE)
# ==========================================

def plot_hdbscan_map(df, cluster_col, num_samples=1000):
    m = folium.Map(location=[39.8283, -98.5795], zoom_start=4, tiles='CartoDB positron')
    
    # Define Colors: Core Clusters vs Noise
    # We use distinct colors for clusters, and GREY/BLACK for Noise
    cluster_config = {
        0: {'color': '#E31A1C', 'name': 'Core Cluster 0 (Red)'},
        1: {'color': '#1F78B4', 'name': 'Core Cluster 1 (Blue)'},
        # Add more if HDBSCAN found more clusters
        -1: {'color': '#666666', 'name': 'Noise / Outliers (Grey)'} # Noise is Grey
    }
    
    layers = {}
    for cid, config in cluster_config.items():
        layers[cid] = folium.FeatureGroup(name=config['name'])

    sample_df = df.sample(n=min(num_samples, len(df)), random_state=42)
    
    for idx, row in sample_df.iterrows():
        cluster = int(row[cluster_col])
        
        if cluster not in cluster_config: continue
            
        folium.CircleMarker(
            location=[row['merch_lat'], row['merch_long']],
            radius=3,
            color=cluster_config[cluster]['color'],
            fill=True,
            fill_opacity=0.6,
            popup=f"Cluster: {cluster}"
        ).add_to(layers[cluster])
        
    for layer in layers.values():
        layer.add_to(m)
        
    folium.LayerControl(collapsed=False).add_to(m)
    return m

# --- EXECUTION ---
print("Generating HDBSCAN Map... (Look for the Grey Noise points)")

# Ensure HDBSCAN Labels are in the main df
if 'Cluster_HDBSCAN' not in df.columns:
    df['Cluster_HDBSCAN'] = df['cc_num'].map(features_df['Cluster_HDBSCAN'])

# Drop NaNs (customers who were filtered out during cleaning)
plot_df = df.dropna(subset=['Cluster_HDBSCAN'])

map_hdb = plot_hdbscan_map(plot_df, 'Cluster_HDBSCAN')
map_hdb.save("map_hdbscan.html")
map_hdb

**Observation:**
A comparative geospatial analysis of the K-Means partitioning (Figure X) versus HDBSCAN density estimation (Figure Y) reveals a complex, **Geographically Agnostic** structure.

* **Visual Evidence (Persona Distribution):** The K-Means projection demonstrates that "Wealthy VIPs" (Cluster 0) and "Budget Night Owls" (Cluster 1) are distributed relatively evenly across the same geographic regions as "Mainstream Families" (Cluster 2). There is no distinct regional clustering (e.g., "East Coast vs. West Coast" divides), invalidating the hypothesis that lifestyle segments are geographically determined.
* **Structural Contrast (Density vs. Noise):** While the K-Means map assigns a persona to every transaction, the HDBSCAN map reveals that **30.2% of geospatial activity is classified as 'Noise' (Grey)**. This indicates that while broad behavioral personas exist nationwide, a significant portion of transaction volume is spatially erratic and lacks the density required to form stable "cores."
* **Conclusion:** The primary drivers of customer differentiation are **Spending Power** and **Day-Night Habits**, not **Physical Mobility**. This implies that marketing strategies can be deployed nationally without region-specific adjustments; however, the 30% "Noise" identified by HDBSCAN suggests that a significant minority of customers exhibit unpredictable spatial patterns, requiring distinct risk-management protocols compared to the stable core segments.

### 6.2 Correlation Analysis & Feature Interaction

To ensure the stability of the clustering model, we examined the pairwise correlations between engineered features. 

In [ ]:
# ==========================================
# CORRELATION ANALYSIS 
# ==========================================

plt.figure(figsize=(12, 10))
correlation = features_df.corr()
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix')
plt.show()

#Text based output 

# 1. Calculate the Matrix
correlation = features_df.corr()

# 2. Unstack to get pairs
# Convert the matrix into a long list of pairs
corr_pairs = correlation.unstack()

# 3. Create a DataFrame to handle sorting
# We need two columns: one for the real value, one for sorting
ranked_corr = pd.DataFrame(corr_pairs, columns=['Signed_Value'])
ranked_corr['Abs_Value'] = ranked_corr['Signed_Value'].abs()

# 4. Filter out self-correlations (where correlation is exactly 1.0)
ranked_corr = ranked_corr[ranked_corr['Abs_Value'] < 1.0]

# 5. Sort by Strength (Absolute Value)
ranked_corr = ranked_corr.sort_values(by='Abs_Value', ascending=False)

# 6. Show Strongest (Top 20 to see the pairs)
print("TOP 10 STRONGEST CORRELATIONS ")
print("--------------------------------------")
# We only print the 'Signed_Value' column
print(ranked_corr['Signed_Value'].head(20))

# 7. Show Weakest (Independent features)
print("\nTOP 10 WEAKEST CORRELATIONS (Near Zero)")
print("---------------------------------------")
print(ranked_corr['Signed_Value'].tail(10))

The analysis reveals three dominant structural relationships that define the customer base.

**1. The "Velocity" Complex (High Multicollinearity)**
* **Observation:** We observed extremely high correlations between `txn_count`, `amt_sum`, and `cat_entropy` (Pearson $r > 0.89$).
* **Interpretation:** This confirms that **Frequency is the master variable** for engagement. Customers who transact frequently (`txn_count`) almost inevitably accumulate higher total spend (`amt_sum`) and visit a wider variety of merchants (`cat_entropy`).
* **Scientific Note:** While such high multicollinearity can be problematic for regression models, in clustering (K-Means), it acts as a **signal amplifier**, reinforcing the "High-Engagement" vs. "Low-Engagement" separation observed in PC1.

**2. The "Small Ticket" Dynamic**
* **Observation:** `amt_mean` (Average Transaction Size) has a strong *negative* correlation with `txn_count` ($r = -0.88$) and `cat_entropy` ($r = -0.92$).
* **Interpretation:** This quantifies the **"Quality vs. Quantity"** trade-off. High-frequency users are typically making small, daily purchases (Coffee, Gas, Lunch). Conversely, low-frequency users are making large, "bulky" purchases (Luxury items, Electronics). This mathematically proves that **you cannot optimize for both** Frequency and Ticket Size simultaneously; they are opposing behavioral forces.

**3. The Geospatial Redundancy**
* **Observation:** `dist_km_mean` and `radius_of_gyration` are highly correlated ($r = 0.87$).
* **Interpretation:** Both metrics measure spatial dispersion. Since they move in lockstep, it confirms that customers who transact far from home (High Distance) are also the ones roaming over a wide area (High Radius). This validates the robustness of our "Mobility" dimension (PC2).

**4. Cluster Drivers (The "Why" behind the Segments)**
* **Observation:** The cluster labels (`Cluster_KMeans`) show the strongest correlations with `travel_ratio` ($r = -0.71$) and `gas_ratio` ($r = +0.59$).
* **Interpretation:** This confirms that **Category Mix** (specifically Travel vs. Commuting) was a primary decision boundary for the K-Means algorithm, even more so than wealth metrics. This validates our decision to engineer ratio-based features, as they ended up being the strongest discriminators of customer persona.

### 6.3 Dynamic Stability Analysis (Transition Matrix)

To assess the temporal consistency of customer value, we analyzed the migration of 908 customers between spending quartiles from Q1 (Jan-Mar) to Q4 (Oct-Dec). The resulting **Transition Matrix** reveals a "Sticky Extremes" phenomenon.


In [ ]:
# ==========================================
# ANALYZE CUSTOMER MOVEMENT (Transitions)
# ==========================================

# 1. Ensure 'trans_dt' exists
if 'trans_dt' not in df.columns:
    df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'])

# 2. Split Data (Q1 vs Q4)
df['month'] = df['trans_dt'].dt.month
early_df = df[df['month'] <= 3]  # Q1
late_df = df[df['month'] >= 10]  # Q4

# 3. Calculate Spending Power (Total Spend)
q1_spend = early_df.groupby('cc_num')['amt'].sum()
q4_spend = late_df.groupby('cc_num')['amt'].sum()

# 4. Align Indices (Only customers present in BOTH periods)
common_customers = q1_spend.index.intersection(q4_spend.index)
q1_val = q1_spend[common_customers]
q4_val = q4_spend[common_customers]

print(f"Analyzing stability for {len(common_customers)} customers who were active in both Q1 and Q4.")

# 5. Bin into Quartiles
q1_bins = pd.qcut(q1_val, 4, labels=['Low', 'Med', 'High', 'V.High'])
q4_bins = pd.qcut(q4_val, 4, labels=['Low', 'Med', 'High', 'V.High'])

# 6. Create Transition Matrix
# normalize='index' means rows sum to 100%
trans_matrix = pd.crosstab(q1_bins, q4_bins, normalize='index')

# --- TEXT OUTPUT (Copy this for your report) ---
print("\n" + "="*40)
print("TRANSITION MATRIX (Stability Analysis)")
print("Rows = Start Status (Q1), Cols = End Status (Q4)")
print("="*40)
# Multiply by 100 to show percentages like "85.2%"
print((trans_matrix * 100).round(1).astype(str) + '%')
print("="*40 + "\n")

# 7. Plot
plt.figure(figsize=(8, 6))
sns.heatmap(trans_matrix, annot=True, fmt=".0%", cmap="Blues")
plt.title("Stability Analysis: Do High Spenders STAY High Spenders?")
plt.ylabel("Status in Q1 (Jan-Mar)")
plt.xlabel("Status in Q4 (Oct-Dec)")
plt.show()

**1. The "Sticky Extremes" Phenomenon (High Retention)**
* **Observation:** Retention rates are highest at the boundaries of the spending spectrum. **83.7%** of "Low Value" customers and **77.5%** of "Very High Value" customers remained in their respective quartiles after 9 months.
* **Interpretation:** This indicates a high degree of behavioral inertia. The bank's most valuable clients ("V.High") are effectively locked in, with negligible risk of drastic churn (0.0% dropped to Low status). Similarly, "Low Value" customers are static, suggesting that passive "activation" campaigns may yield limited ROI without significant incentives.

**2. Volatility in the "Aspiring" Segment (The High-Value Churn Risk)**
* **Observation:** The "High Value" (Tier 3) segment exhibits the lowest stability (**59.0% retention**). This group is equally likely to "Graduate" to VIP status (**19.8%**) as they are to "Downgrade" to Medium status (**19.8%**).
* **Strategic Implication:** This volatility identifies the **"High Value" segment as the critical battleground**. Unlike the stable VIPs, these customers are at a tipping point. Marketing resources should be disproportionately allocated here—specifically **retention offers** to prevent the 19.8% downgrade and **upsell incentives** to accelerate the 19.8% graduation.

**3. Organic Growth Pipeline**
* **Observation:** Upward mobility is visible in the "Medium" tier, where **18.9%** of customers grew into the "High" tier organically.
* **Conclusion:** There is a healthy natural pipeline of customer maturation, validating the potential for "Share of Wallet" expansion strategies among middle-tier users.

### 6.4 Decision Tree

In [ ]:
# Train a simple tree to explain the clusters
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
X = features_df.drop(columns=['Cluster_KMeans', 'Cluster_HDBSCAN'], errors='ignore')
y = features_df['Cluster_KMeans']

clf.fit(X, y)

plt.figure(figsize=(20, 10))
plot_tree(clf, feature_names=X.columns, class_names=[f'C{i}' for i in sorted(y.unique())], filled=True, fontsize=10)
plt.title("Cluster Decision Rules (How to classify a new customer)")
plt.show()



### 6.4 Surrogate Model Interpretation (Decision Rules)

To operationalize the clustering results for the marketing team, we trained a Decision Tree to extract the primary "If-Then" rules that define each persona. The model highlights three critical thresholds:

  * **Average Transaction Value Threshold:** \~$94 (`log_amt_mean` ≈ 4.55)
  * **Travel Spend Threshold:** 11% of Wallet (`travel_ratio` ≈ 0.11)
  * **Wealth Threshold:** High vs. Low Total Spend (`log_amt_sum` ≈ 10.02)

**The Classification Logic:**

#### **1. The Mainstream Family (Cluster 2)**

  * **The Rule:** `Average Transaction Size <= $94`
  * **Business Interpretation:** If a customer's average purchase is low, they are almost certainly "Mainstream."
  * **Why:** This confirms that this segment is defined by **High Frequency / Low Value** transactions. They use the card for daily "micro-payments" (Coffee, Gas, Groceries), which drags their average ticket size down.

#### **2. The Wealthy VIP (Cluster 0)**

  * **The Rule:** `Average Transaction > $94` **AND** `Travel Spend > 11%`
  * **Business Interpretation:** If a customer makes large purchases **and** allocates more than 11% of their budget to travel categories, they are a VIP.
  * **Why:** This validates the "Jetsetter" persona. They are not buying daily coffee (which would lower their average); they are buying flight tickets and hotel stays.

#### **3. The Budget Night Owl (Cluster 1)**

  * **The Rule:** `Average Transaction > $94` **AND** `Total Spend < Low Threshold`
  * **Business Interpretation:** If a customer makes "big" purchases but has a **low total spend**, they are the Niche/Budget group.
  * **Why:** This seems contradictory (High Ticket, Low Total) but mathematically implies **Low Frequency**. This customer swipes the card very rarely (perhaps once a month for a specific bill or online purchase), so their average is high, but their total value to the bank is low.



### **7. Operational Implementation Strategy**

To maximize the ROI of this segmentation, we propose a two-pronged strategy covering both the retention of existing customers and the acquisition of new ones.

**7.1 Retention & Upsell (Existing Customers)**
Based on the Decision Tree rules derived in Section 6.4, we propose a simplified logic for real-time customer classification:

1.  **Step 1 (The "Daily Use" Filter):** Calculate the customer's Average Transaction Value (ATV).
    * *If ATV < $95:* Flag as **"Mainstream"**. Deploy retention strategies (e.g., Cashback on Essentials) to maintain habit.
2.  **Step 2 (The "Travel" Filter):** For remaining customers, check Travel Spend % (Airlines/Hotels).
    * *If Travel > 11%: * Flag as **"VIP"**. Deploy premium upsell strategies (e.g., Travel Insurance, Concierge Service).
3.  **Step 3 (The "Frequency" Filter):** For the remainder, check Total Spend Volume.
    * *If Volume is Low:* Flag as **"Niche/Low-Frequency"**. Deploy activation strategies (e.g., "Spend X get Y" challenges) to increase transaction velocity.

**7.2 Acquisition (New Customers)**
While this model segments based on behavior, these clusters can guide **Lookalike Audience** targeting for marketing.
* **Strategy:** By analyzing the demographic profile (Age, Location, Job) of the high-value **"VIP"** cluster, the marketing team can target non-customers who share these traits.
* **Benefit:** This allows the bank to target new users based on the *predicted* likelihood of them becoming high-value spenders, rather than generic mass marketing.


## 8. Conclusion

This analysis successfully transformed **1.3 million raw transaction records** into a streamlined segmentation model of **983 unique customer profiles**. By moving beyond simple aggregate metrics and utilizing behavioral feature engineering (e.g., Radius of Gyration, Shannon Entropy, Temporal Ratios), we identified three distinct customer personas that drive the bank's portfolio.

### **8.1 Key Findings**

| Segment | Cluster | Key Characteristic | Revenue Driver | Strategic Action |
| :--- | :--- | :--- | :--- | :--- |
| **Wealthy VIP** | 0 | High Spend + Travel | Margin | Premium Upsell |
| **Budget Night Owl** | 1 | Off-hours, Low Total Spend | Potential Churn | Reactivation |
| **Mainstream Family** | 2 | High Freq + Low Value | Volume | Habit Retention |

The K-Means clustering algorithm, validated by the Elbow Method and cross-checked by GMM and HDBSCAN, revealed that the customer base falls into three clear behavioral patterns:

* **Cluster 0 — The Wealthy VIP:** High total spend concentrated in travel and high-ticket non-essentials; low engagement with daily-essential categories (gas, groceries). Minority of users, majority of margin.
* **Cluster 1 — The Budget Night Owl:** Transacts almost exclusively during non-standard hours with low total volume. Churn-risk segment.
* **Cluster 2 — The Mainstream Family:** Consistent, low-value daily transactions (groceries, gas, dining). Volume backbone of the portfolio.

### **8.2 Operational Value**

Crucially, the complex K-Means model was translated into a transparent **Rule-Based System** (Section 6.4) suitable for real-time deployment. By monitoring just two metrics — **Average Transaction Value (ATV ~\$94)** and **Travel Spend Percentage (~11%)** — the bank can classify any customer without retraining the model or running an ML runtime in production.

### **8.3 Strategic Implications**

The segmentation enables a shift from generic mass-marketing to a targeted three-pillar strategy: **Retention** of the Wealthy VIP tier via premium upsells, **Reactivation** of the Budget Night Owl tier via activation incentives, and **Habit Retention** of the Mainstream Family tier via cashback on essentials. The Q1→Q4 Transition Matrix (§6.3) further localises the marketing battleground to the High-Value quartile, where 19.8% of customers risk downgrading and an equal 19.8% are primed to graduate to VIP status.
